# **1. Extraction, Transformation, and Loading Phase:**

In [2]:
import pandas as pd
import numpy as np

---

## **1.1. Data Extraction**

In [11]:
transactions = pd.read_csv('../data/raw/transactions_raw.csv')          # ...capitalize

stores = pd.read_csv('../data/raw/stores_raw.csv')                      # ...capitalize

products = pd.read_csv('../data/raw/products_raw.csv')                  # ...

customers = pd.read_csv('../data/raw/customers_raw.csv')                # ...

---

## **1.2. Data Transformation**

### 1.2.1. Transform the Datasets

In [13]:
transactions.columns = transactions.columns.str.lower()

transactions['date'] = pd.to_datetime(transactions['date'], errors='coerce', dayfirst=False)        # Format date column consistantly

transactions = transactions.dropna(subset='date')                                                   # drop all null data points

transactions = transactions[(transactions['qty'] > 0) & (transactions['unit_price'] > 0)]           # remove bad row (negative datapoints)

transactions['revenue'] = (transactions['qty'] * transactions['unit_price']).round(2)               # create a revenue column

transactions['promo_applied'] = transactions['promo_applied'].astype(int)

In [14]:
transactions.head().dtypes

txn_id                    object
date              datetime64[ns]
store_id                  object
product_id                object
qty                        int64
unit_price               float64
promo_applied              int64
channel                   object
payment_method            object
cust_id                   object
revenue                  float64
dtype: object

### 1.2.2. Merge the Dataframes

In [15]:
fact = (
    transactions.merge(products[['product_id', 'category', 'brand', 'list_price']], on='product_id', how='left')
    .merge(stores[['store_id', 'store_name', 'region', 'city']], on='store_id', how='left')
    .merge(customers[['cust_id', 'gender', 'age', 'region', 'city']].rename(columns={'region':'cust_region', 'city':'cust_city'}), on='cust_id', how='left')
)

In [16]:
fact = fact[["txn_id", "date", "store_id", "product_id", "qty", 
             "unit_price", "promo_applied", "channel", "payment_method", 
             "cust_id", "gender", "age", "cust_region", "cust_city", "revenue"]]

In [ ]:
fact['revenue']

In [17]:
fact = fact.where(pd.notna(fact), None)
fact.to_csv('../data/processed/fact_sales.csv', index=False)
stores.to_csv('../data/processed/dim_store.csv', index=False)
products.to_csv('../data/processed/dim_product.csv', index=False)
customers.to_csv('../data/processed/dim_customer.csv', index=False)

---

## **1.3. Loading to SQL Server Phase:**

In [18]:
import pyodbc


In [10]:
SERVER = "np:\\\\.\\pipe\\LOCALDB#BDCF2521\\tsql\\query"
DATABASE = "ShopSmartDW"

In [19]:
# conn_str = (
#     "DRIVER={ODBC Driver 17 for SQL Server};"
#     f"SERVER={SERVER};"
#     f"DATABASE={DATABASE};"
#     "Trusted_Connection=yes;"
# )

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=(localdb)\\MSSQLLocalDB;"
    "DATABASE=ShopSmartDW;"
    "Trusted_Connection=yes;"
)

In [21]:
# read processed CSV files

fact = pd.read_csv("../data/processed/fact_sales.csv")
dim_store = pd.read_csv("../data/processed/dim_store.csv")
dim_product = pd.read_csv("../data/processed/dim_product.csv")
dim_customer = pd.read_csv("../data/processed/dim_customer.csv")

In [22]:
# Convert date columns properly
fact["date"] = pd.to_datetime(fact["date"])

fact["age"] = fact["age"].apply(lambda x: None if (isinstance(x, float) and np.isnan(x)) else x)
fact['age'] = fact['age'].fillna(0)

dim_store["opened_date"] = pd.to_datetime(dim_store["opened_date"]).dt.date
dim_customer["signup_date"] = pd.to_datetime(dim_customer["signup_date"]).dt.date

fact['revenue'] = pd.to_numeric(fact['revenue'], errors='coerce')
fact['revenue'] = fact['revenue'].fillna(0)

In [45]:
print(fact.head().dtypes)

txn_id                    object
date              datetime64[ns]
store_id                  object
product_id                object
qty                        int64
unit_price               float64
promo_applied              int64
channel                   object
payment_method            object
cust_id                   object
gender                    object
age                      float64
cust_region               object
cust_city                 object
revenue                  float64
dtype: object


In [ ]:
fact.info()

In [23]:
# Replace NaN with None so SQL receives proper NULLs
fact = fact.where(pd.notna(fact), None)
dim_store = dim_store.where(pd.notna(dim_store), None)
dim_product = dim_product.where(pd.notna(dim_product), None)
dim_customer = dim_customer.where(pd.notna(dim_customer), None)

In [24]:
def insert_dataframe(df, table_name, connection):
    cursor = connection.cursor()
    cursor.fast_executemany = True
    columns = ",".join(df.columns)
    placeholders = ",".join(["?"] * len(df.columns))
    sql = f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})"
    cursor.executemany(sql, df.values.tolist())
    connection.commit()



In [ ]:
fact.head()

In [25]:
with pyodbc.connect(conn_str) as conn:
    # Clear tables (for learning only)
    conn.execute("DELETE FROM dbo.FactSales")
    conn.execute("DELETE FROM dbo.DimStore")
    conn.execute("DELETE FROM dbo.DimProduct")
    conn.execute("DELETE FROM dbo.DimCustomer")
    conn.commit()

    insert_dataframe(dim_store, "dbo.DimStore", conn)
    insert_dataframe(dim_product, "dbo.DimProduct", conn)
    insert_dataframe(dim_customer, "dbo.DimCustomer", conn)
    insert_dataframe(fact, "dbo.FactSales", conn)

---

# **DEBUGGING PHASE**

In [42]:
print(fact.dtypes)

txn_id                    object
date              datetime64[ns]
store_id                  object
product_id                object
qty                        int64
unit_price               float64
promo_applied              int64
channel                   object
payment_method            object
cust_id                   object
gender                    object
age                      float64
cust_region               object
cust_city                 object
revenue                  float64
dtype: object


In [53]:
cursor = conn.cursor()
cursor.execute("SELECT DB_NAME()")
print(cursor.fetchone())

('ShopSmartDW',)


In [ ]:
# Debug - find the problematic row
cursor = conn.cursor()
columns = ",".join(fact.columns)
placeholders = ",".join(["?"] * len(fact.columns))
sql = f"INSERT INTO dbo.FactSales ({columns}) VALUES ({placeholders})"

for i, row in enumerate(fact.values.tolist()):
    try:
        cursor.execute(sql, row)
    except Exception as e:
        print(f"Row {i} failed: {e}")
        print(row)
        break

In [ ]:
# import pandas as pd
# import numpy as np
# import pyodbc

# # Read the CSV files
# print("📂 Loading CSV files...")
# fact = pd.read_csv("processed1/fact_sales.csv")
# dim_store = pd.read_csv("processed1/dim_store.csv")
# dim_product = pd.read_csv("processed1/dim_product.csv")
# dim_customer = pd.read_csv("processed1/dim_customer.csv")

# # ===== CONVERT DATE COLUMNS =====
# print("\n📅 Converting date columns...")
# fact["date"] = pd.to_datetime(fact["date"]).dt.date
# dim_store["opened_date"] = pd.to_datetime(dim_store["opened_date"]).dt.date
# dim_customer["signup_date"] = pd.to_datetime(dim_customer["signup_date"]).dt.date

# # ===== FILL MISSING VALUES WITH DEFAULTS =====
# print("\n🔄 Filling missing values with defaults...")

# # FactSales - fill missing values
# fact['cust_id'] = fact['cust_id'].fillna('UNKNOWN')
# fact['gender'] = fact['gender'].fillna('U')  # U for Unknown
# fact['age'] = fact['age'].fillna(0)
# fact['cust_region'] = fact['cust_region'].fillna('Unknown')
# fact['cust_city'] = fact['cust_city'].fillna('Unknown')

# # Convert age to float (to match SQL FLOAT)
# fact['age'] = fact['age'].astype(float)

# # Ensure promo_applied is integer (0 or 1)
# fact['promo_applied'] = fact['promo_applied'].astype(int)

# print("✅ Missing values filled")

# # ===== IMPROVED INSERT FUNCTION =====
# def insert_dataframe(df, table_name, connection):
#     cursor = connection.cursor()
#     cursor.fast_executemany = True
    
#     columns = ",".join(df.columns)
#     placeholders = ",".join(["?"] * len(df.columns))
#     sql = f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})"
    
#     # Convert to list of tuples
#     data = [tuple(x) for x in df.values]
    
#     try:
#         cursor.executemany(sql, data)
#         connection.commit()
#         print(f"✅ Inserted {len(df)} rows into {table_name}")
#     except Exception as e:
#         print(f"❌ Error inserting into {table_name}: {e}")
#         # Print sample of first row for debugging
#         if len(data) > 0:
#             print("\nFirst row values:")
#             for col, val in zip(df.columns, data[0]):
#                 print(f"  {col}: {repr(val)}")
#         raise e

# # ===== MAIN EXECUTION =====
# print("\n🚀 Connecting to SQL Server...")
# with pyodbc.connect(conn_str) as conn:
#     print("✅ Connected successfully!")
    
#     # Clear tables
#     print("\n🧹 Clearing existing data...")
#     conn.execute("DELETE FROM dbo.FactSales")
#     conn.execute("DELETE FROM dbo.DimStore")
#     conn.execute("DELETE FROM dbo.DimProduct")
#     conn.execute("DELETE FROM dbo.DimCustomer")
#     conn.commit()
    
#     # Insert data
#     print("\n📤 Inserting data...")
#     insert_dataframe(dim_store, "dbo.DimStore", conn)
#     insert_dataframe(dim_product, "dbo.DimProduct", conn)
#     insert_dataframe(dim_customer, "dbo.DimCustomer", conn)
#     insert_dataframe(fact, "dbo.FactSales", conn)
    
#     print("\n✅ All data loaded successfully!")